# 1. Imports

In [2]:
import os
from dotenv import load_dotenv
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus

# 2. Connection

In [3]:
load_vars = load_dotenv("../.env")
if load_vars:
    print("Variáveis de ambiente carregadas com sucesso")
else:
    print("Arquivo .env não encontrado!")

Variáveis de ambiente carregadas com sucesso


In [4]:
connection_string = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=tcp:127.0.0.1,1433;"
    "DATABASE=IHM_Testes_2;"
    "UID=sa;"
    f"PWD={os.environ['MSSQL_SA_PASSWORD']};"
    "TrustServerCertificate=yes;"
    "Encrypt=no;"
)

In [5]:
connection_url = f"mssql+pyodbc:///?odbc_connect={quote_plus(connection_string)}"

In [6]:
engine = create_engine(connection_url)

# 3. Pegando todas tabelas

In [7]:
tables = [
    'dados_receitas',
    'fila_batch_ids',
    'fila_paradas',
    'ihms',
    'linhas_producao',
    'logs_registradores',
    'maqteste_status_geral',
    'notificacoes',
    'ordens_servico',
    'paradas',
    'parametros',
    'receitas',
    'registradores',
    'sistemas'
]

In [8]:
dicionario_dfs = dict()
for table in tables:
    df_apoio = pd.read_sql(f"SELECT * FROM {table}", engine)
    if len(df_apoio) > 0:
        dicionario_dfs[table] = df_apoio
    del df_apoio

In [9]:
# for key in dicionario_dfs.keys():
#     print(key)
#     print(dicionario_dfs[key].head())
#     print('------')
    

# 4. Tratando informação

In [10]:
df_interesse = dicionario_dfs['logs_registradores'].merge(dicionario_dfs['ihms'].rename(columns={'id':'id_ihm'})[['id_ihm', 'nome_maquina']], how='left', on='id_ihm')

In [11]:
df_interesse = df_interesse.merge(dicionario_dfs['registradores'].rename(columns={'id':'id_registrador'})[['id_registrador', 'descricao']], how='left', on='id_registrador')

In [12]:
df_interesse = df_interesse[['nome_maquina', 'descricao', 'datahora', 'valor_bruto']]

In [13]:
df_interesse = df_interesse.pivot_table(index=['nome_maquina','datahora'], columns='descricao', values='valor_bruto', aggfunc='first').reset_index()

In [14]:
df_interesse = df_interesse.sort_values('datahora')
df_interesse.reset_index(drop=True, inplace=True)

In [15]:
df_interesse.columns

Index(['nome_maquina', 'datahora', 'engenheiro', 'manutentor', 'motivo_parada',
       'operador', 'produzido', 'reprovado', 'status_maquina',
       'total_produzido'],
      dtype='object', name='descricao')

In [16]:
relatorio = "Iniciando relatório..."
colunas_interesse = ['engenheiro', 'manutentor', 'motivo_parada',
       'operador', 'produzido', 'reprovado', 'status_maquina',
       'total_produzido']
dicionario_controle = dict()
for i, row in df_interesse.iterrows():
    relatorio += f"\n\n**{row['datahora']}**" 
    if i > 0:
        relatorio += "\nMudanças de Estado"
    for col in colunas_interesse:
        if col not in dicionario_controle.keys():
            relatorio += f"\n{col} (Estado inicial): {row[col]}"
            dicionario_controle[col] = row[col]
        else:
            if dicionario_controle[col] != row[col]:
                relatorio += f"\n{col}: {row[col]}"
                dicionario_controle[col] = row[col]

In [17]:
# print(relatorio)

In [18]:
df_interesse

descricao,nome_maquina,datahora,engenheiro,manutentor,motivo_parada,operador,produzido,reprovado,status_maquina,total_produzido
0,MAQ1,2025-11-15 18:07:08.973,0,0,0,0,43,21,0,64
1,MAQ1,2025-11-15 18:07:39.020,0,0,5,0,43,21,0,64
2,MAQ1,2025-11-15 18:07:40.570,0,0,5,0,43,21,5,64
3,MAQ1,2025-11-15 18:09:06.320,0,0,0,0,43,21,0,64
4,MAQ1,2025-11-15 18:09:21.357,0,0,0,7,43,21,0,64
5,MAQ1,2025-11-15 18:09:37.660,0,0,49,7,43,21,49,64
6,MAQ1,2025-11-15 18:10:05.793,0,0,0,7,43,21,0,64
7,MAQ1,2025-11-15 18:16:11.040,0,0,2,7,43,21,2,64
8,MAQ1,2025-11-15 18:16:20.453,0,0,49,7,43,21,49,64
9,MAQ1,2025-11-15 18:16:21.257,0,0,0,7,43,21,0,64


In [ ]:
def get_metrics_machine(df_timeline, machine='MAQ1'):
    # OEE = Disponibilidade * Performance * Qualidade
    # Disponibilidade = Tempo produzido / Tempo programado para produzir
    # Performance = Produção Real / Produção Teórica
    # Qualidade = Peça Boas / Total de peças
    
    # OEE, Qualidade, Eficiencia, Meta, Acumulado, Operador, Manutentor, Status
    
    
    
    